<a href="https://colab.research.google.com/github/keksenia/cstati-event-analytics/blob/main/notebooks/02_identity_resolution.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 02 — Identity Resolution

Цель ноутбука — построить private-логику связывания участников между мероприятиями cstati.

В этом ноутбуке мы:

- загружаем raw CSV и manual-справочники;
- определяем поля, похожие на ФИО, Telegram, телефон, email, группу и программу;
- нормализуем идентификаторы;
- строим внутренний `participant_key_internal`;
- присваиваем уровень уверенности `identity_confidence`;
- проверяем качество stitching;
- сохраняем private-слой для следующих ноутбуков.

Важно: этот ноутбук работает с персональными данными.  
Файлы из `data/interim_private/` нельзя публиковать в публичном GitHub.

In [141]:
from pathlib import Path
import re
import os
import hmac
import hashlib
import unicodedata
import warnings

import pandas as pd
import numpy as np

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)

PROJECT_ROOT = Path("/content")
RAW_DIR = Path("/content")
MANUAL_DIR = PROJECT_ROOT / "manual"

IDENTITY_DIR = PROJECT_ROOT / "data" / "interim_private" / "identity"
IDENTITY_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RAW_DIR:", RAW_DIR)
print("MANUAL_DIR:", MANUAL_DIR)
print("IDENTITY_DIR:", IDENTITY_DIR)

PROJECT_ROOT: /content
RAW_DIR: /content
MANUAL_DIR: /content/manual
IDENTITY_DIR: /content/data/interim_private/identity


In [142]:
from pathlib import Path

Path("/content/manual").mkdir(parents=True, exist_ok=True)

In [143]:
from pathlib import Path

MANUAL_DIR = Path("/content/manual")
MANUAL_DIR.mkdir(parents=True, exist_ok=True)

aliases_text = """raw_event_name,canonical_event_id,canonical_event_name,source,notes
Посвят'23,posvyat_2023,Посвят'23,raw/svodnaya,
Антипосвят'23,antiposvyat_2023,Антипосвят'23,raw/svodnaya,
Бал ФКН'24,ball_fkn_2024,Бал ФКН'24,raw/svodnaya,
ANNIVERSARY'24,anniversary_2024,ANNIVERSARY'24,raw/svodnaya,
Ballmer Peak'24,ballmer_peak_2024,Ballmer Peak'24,raw/svodnaya,
Коллаб'24,collab_2024,Коллаб'24,raw/svodnaya,
Настолки'24,nastolki_2024,Настолки'24,raw/svodnaya,
Нейрорейв,neyrorave_2024,Нейрорейв,raw/svodnaya,
Поход'24,pohod_2024,Поход'24,raw/svodnaya,
Экватор'24,ekvator_2024,Экватор'24,raw/svodnaya,
GLANZ,glanz_2024,GLANZ,raw/svodnaya,
ЗВ'25,zv_2025,ЗВ'25,raw,Проверить соответствие со сводной
ЗВ'24,zv_2025,ЗВ'25,svodnaya,Возможный alias - нужно подтвердить
Посвят'25,posvyat_2025,Посвят'25,raw/svodnaya,
Антипосвят'25,antiposvyat_2025,Антипосвят'25,raw/svodnaya,
CSFEST'25,csfest_2025,CSFEST'25,raw,Есть raw-файл но отсутствует в Сводной
Поход'25,pohod_2025,Поход'25,raw/svodnaya,
Экватор'25,ekvator_2025,Экватор'25,raw/svodnaya,
ЗВ'26,zv_2026,ЗВ'26,raw/svodnaya,
"""

(MANUAL_DIR / "event_aliases.csv").write_text(aliases_text, encoding="utf-8")

aliases_test = pd.read_csv(MANUAL_DIR / "event_aliases.csv", dtype=str)
print(aliases_test.shape)
print(aliases_test.columns.tolist())
display(aliases_test.head())

(19, 5)
['raw_event_name', 'canonical_event_id', 'canonical_event_name', 'source', 'notes']


,raw_event_name,canonical_event_id,canonical_event_name,source,notes
0,Посвят'23,posvyat_2023,Посвят'23,raw/svodnaya,NaN
1,Антипосвят'23,antiposvyat_2023,Антипосвят'23,raw/svodnaya,NaN
2,Бал ФКН'24,ball_fkn_2024,Бал ФКН'24,raw/svodnaya,NaN
3,ANNIVERSARY'24,anniversary_2024,ANNIVERSARY'24,raw/svodnaya,NaN
4,Ballmer Peak'24,ballmer_peak_2024,Ballmer Peak'24,raw/svodnaya,NaN


In [144]:
def read_csv_robust(path: Path, nrows=None) -> pd.DataFrame:
    encodings = ["utf-8-sig", "utf-8", "cp1251"]
    last_error = None

    for encoding in encodings:
        try:
            return pd.read_csv(
                path,
                dtype=str,
                encoding=encoding,
                sep=None,
                engine="python",
                nrows=nrows,
                on_bad_lines="skip",
            )
        except Exception as e:
            last_error = e

    raise RuntimeError(f"Не удалось прочитать {path.name}: {last_error}")


def load_manual_csv(filename: str) -> pd.DataFrame:
    path = MANUAL_DIR / filename

    if not path.exists():
        print(f"Нет файла: {path}")
        return pd.DataFrame()

    for sep in [",", ";"]:
        try:
            df = pd.read_csv(path, dtype=str, sep=sep)
            df.columns = [c.strip() for c in df.columns]
            if len(df.columns) > 1:
                return df
        except Exception:
            pass

    df = pd.read_csv(path, dtype=str, sep=None, engine="python")
    df.columns = [c.strip() for c in df.columns]
    return df


def clean_filename_event_name(path: Path) -> str:
    name = path.stem
    prefix = "Копия Анализ аудитории - "

    if name.startswith(prefix):
        name = name[len(prefix):]

    return name.strip()


raw_files = sorted(RAW_DIR.glob("*.csv"))

# исключаем manual-файлы, если они случайно лежат в /content
raw_files = [
    f for f in raw_files
    if f.name not in {
        "event_metadata.csv",
        "event_aliases.csv",
        "attendance_corrections.csv",
    }
]

metadata = load_manual_csv("event_metadata.csv")
aliases = load_manual_csv("event_aliases.csv")
attendance_corrections = load_manual_csv("attendance_corrections.csv")

print("raw files:", len(raw_files))
print("metadata:", metadata.shape)
print("aliases:", aliases.shape)
print("attendance_corrections:", attendance_corrections.shape)

for f in raw_files:
    print("-", clean_filename_event_name(f))


raw files: 20
metadata: (18, 13)
aliases: (19, 5)
attendance_corrections: (1, 7)
- ANNIVERSARY'24
- Ballmer Peak'24
- CSFEST'25
- GLANZ
- Антипосвят'23
- Антипосвят'25
- Бал ФКН'24
- ЗВ'25
- ЗВ'26
- Коллаб'24
- Мероприятия
- Настолки'24
- Нейрорейв
- Посвят'23
- Посвят'25
- Поход'24
- Поход'25
- Сводная
- Экватор'24
- Экватор'25


In [145]:
AUXILIARY_SOURCE_NAMES = {
    "Сводная",
    "Мероприятия",
}


def normalize_text_base(x) -> str | None:
    if pd.isna(x):
        return None

    x = str(x)
    x = unicodedata.normalize("NFC", x)
    x = x.replace("\u00a0", " ")
    x = x.strip()

    if x == "":
        return None

    return x


def normalize_event_name_for_match(x: str) -> str:
    x = normalize_text_base(x)

    if x is None:
        return ""

    x = x.lower().replace("ё", "е")
    x = re.sub(r"\s+", " ", x)

    return x


aliases_fixed = aliases.copy()
aliases_fixed["raw_event_name_norm"] = aliases_fixed["raw_event_name"].apply(
    normalize_event_name_for_match
)

event_alias_map = aliases_fixed.set_index("raw_event_name_norm").to_dict("index")


def resolve_event_name(raw_event_name: str) -> dict:
    raw_norm = normalize_event_name_for_match(raw_event_name)

    if raw_event_name in AUXILIARY_SOURCE_NAMES:
        return {
            "raw_event_name": raw_event_name,
            "raw_event_name_norm": raw_norm,
            "canonical_event_id": None,
            "canonical_event_name": None,
            "is_auxiliary_source": True,
            "alias_status": "auxiliary_source_not_event",
        }

    if raw_norm in event_alias_map:
        row = event_alias_map[raw_norm]
        return {
            "raw_event_name": raw_event_name,
            "raw_event_name_norm": raw_norm,
            "canonical_event_id": row.get("canonical_event_id"),
            "canonical_event_name": row.get("canonical_event_name"),
            "is_auxiliary_source": False,
            "alias_status": "matched",
        }

    return {
        "raw_event_name": raw_event_name,
        "raw_event_name_norm": raw_norm,
        "canonical_event_id": None,
        "canonical_event_name": None,
        "is_auxiliary_source": False,
        "alias_status": "missing_alias",
    }


event_resolution_check = pd.DataFrame([
    resolve_event_name(clean_filename_event_name(f))
    for f in raw_files
])

event_resolution_check

,raw_event_name,raw_event_name_norm,canonical_event_id,canonical_event_name,is_auxiliary_source,alias_status
0,ANNIVERSARY'24,anniversary'24,anniversary_2024,ANNIVERSARY'24,False,matched
1,Ballmer Peak'24,ballmer peak'24,ballmer_peak_2024,Ballmer Peak'24,False,matched
2,CSFEST'25,csfest'25,csfest_2025,CSFEST'25,False,matched
3,GLANZ,glanz,glanz_2024,GLANZ,False,matched
4,Антипосвят'23,антипосвят'23,antiposvyat_2023,Антипосвят'23,False,matched
5,Антипосвят'25,антипосвят'25,antiposvyat_2025,Антипосвят'25,False,matched
6,Бал ФКН'24,бал фкн'24,ball_fkn_2024,Бал ФКН'24,False,matched
7,ЗВ'25,зв'25,zv_2025,ЗВ'25,False,matched
8,ЗВ'26,зв'26,zv_2026,ЗВ'26,False,matched
9,Коллаб'24,коллаб'24,collab_2024,Коллаб'24,False,matched


In [146]:
def normalize_colname(col: str) -> str:
    col = str(col)
    col = unicodedata.normalize("NFC", col)
    col = col.strip().lower().replace("ё", "е")
    col = re.sub(r"\s+", " ", col)
    return col


def is_unnamed_column(col: str) -> bool:
    return normalize_colname(col).startswith("unnamed")


IDENTIFIER_PATTERNS = {
    "fio": [
        r"^фио$",
        r"фамил",
        r"^имя$",
        r"отчеств",
        r"full name",
        r"first name",
        r"last name",
        r"surname",
    ],
    "telegram": [
        r"telegram",
        r"телеграм",
        r"^tg$",
        r"tg @",
        r"^тг$",
        r"ник в телеграм",
        r"никнейм",
        r"nickname",
        r"username",
        r"t\.me",
    ],
    "phone": [
        r"телефон",
        r"номер телефона",
        r"phone",
        r"mobile",
    ],
    "email": [
        r"^email$",
        r"e-mail",
        r"почта",
        r"mail",
    ],
    "group": [
        r"группа",
        r"^group$",
    ],
    "program": [
        r"направ",
        r"программа",
        r"program",
        r"faculty",
        r"факультет",
    ],
    "course": [
        r"^курс$",
        r"^year$",
    ],
}


def detect_identifier_columns(df: pd.DataFrame) -> dict:
    detected = {k: [] for k in IDENTIFIER_PATTERNS}

    for col in df.columns:
        col_norm = normalize_colname(col)

        # Не считаем Unnamed-колонки ФИО/именем автоматически.
        # Такие файлы потом лучше парсить special loader'ом.
        if is_unnamed_column(col):
            continue

        for role, patterns in IDENTIFIER_PATTERNS.items():
            if any(re.search(pattern, col_norm) for pattern in patterns):
                detected[role].append(col)

    return detected


identifier_column_inventory = []

for path in raw_files:
    event_name = clean_filename_event_name(path)
    df = read_csv_robust(path)
    detected = detect_identifier_columns(df)

    row = {
        "source_file": path.name,
        "inferred_event_name": event_name,
    }

    for role, cols in detected.items():
        row[f"{role}_columns"] = " | ".join(cols)
        row[f"{role}_n_columns"] = len(cols)

    identifier_column_inventory.append(row)

identifier_column_inventory_df = pd.DataFrame(identifier_column_inventory)

display(identifier_column_inventory_df)

,source_file,inferred_event_name,fio_columns,fio_n_columns,telegram_columns,telegram_n_columns,phone_columns,phone_n_columns,email_columns,email_n_columns,group_columns,group_n_columns,program_columns,program_n_columns,course_columns,course_n_columns
0,Копия Анализ аудитории - ANNIVERSARY'24.csv,ANNIVERSARY'24,ФИО,1,Телеграм,1,,0,,0,,0,,0,,0
1,Копия Анализ аудитории - Ballmer Peak'24.csv,Ballmer Peak'24,,0,Tg @,1,,0,,0,,0,Направление,1,,0
2,Копия Анализ аудитории - CSFEST'25.csv,CSFEST'25,Фамилия | Имя | Отчество,3,Телеграм,1,,0,Email,1,Группа,1,,0,,0
3,Копия Анализ аудитории - GLANZ.csv,GLANZ,ФИО,1,Телеграм,1,,0,,0,,0,Факультет,1,,0
4,Копия Анализ аудитории - Антипосвят'23.csv,Антипосвят'23,ФИО,1,Телеграм,1,Телефон,1,,0,,0,Направление,1,Курс,1
5,Копия Анализ аудитории - Антипосвят'25.csv,Антипосвят'25,ФИО,1,Телеграм,1,Телефон,1,,0,,0,,0,Курс,1
6,Копия Анализ аудитории - Бал ФКН'24.csv,Бал ФКН'24,ФИО,1,,0,,0,,0,,0,,0,,0
7,Копия Анализ аудитории - ЗВ'25.csv,ЗВ'25,ФИО,1,Телеграм,1,Телефон,1,,0,,0,Программа,1,Курс,1
8,Копия Анализ аудитории - ЗВ'26.csv,ЗВ'26,Имя | Фамилия,2,Телеграм,1,,0,,0,,0,,0,,0
9,Копия Анализ аудитории - Коллаб'24.csv,Коллаб'24,,0,,0,,0,,0,,0,,0,,0


In [147]:
EMPTY_TOKENS = {
    "",
    "-",
    "—",
    "нет",
    "не знаю",
    "не указано",
    "не указан",
    "nan",
    "none",
    "null",
    "0",
}


def clean_raw_value(x):
    x = normalize_text_base(x)

    if x is None:
        return None

    x_low = x.lower().strip()

    if x_low in EMPTY_TOKENS:
        return None

    return x


def coalesce_values(row: pd.Series, columns: list[str]) -> str | None:
    values = []

    for col in columns:
        if col in row.index:
            v = clean_raw_value(row[col])
            if v is not None:
                values.append(v)

    if not values:
        return None

    return " ".join(values)


def norm_fio(x):
    x = clean_raw_value(x)

    if x is None:
        return None

    x = unicodedata.normalize("NFC", x)
    x = x.lower().replace("ё", "е")
    x = re.sub(r"[^a-zа-я\s-]", " ", x)
    x = re.sub(r"\s+", " ", x).strip()

    if len(x) < 2:
        return None

    return x

def fio_signature(x):
    fio = norm_fio(x)

    if fio is None:
        return None

    tokens = fio.split()

    # Убираем слишком короткие токены
    tokens = [t for t in tokens if len(t) > 1]

    if not tokens:
        return None

    # Сортируем, чтобы "Иванов Иван" и "Иван Иванов" стали одинаковыми
    tokens = sorted(tokens)

    return " ".join(tokens)

def norm_telegram(x):
    x = clean_raw_value(x)

    if x is None:
        return None

    x = unicodedata.normalize("NFC", x)
    x = x.lower().strip()

    x = x.replace("https://", "").replace("http://", "")
    x = x.replace("t.me/", "")
    x = x.replace("telegram.me/", "")
    x = x.replace("@", "")
    x = x.strip()

    x = re.sub(r"[^a-z0-9_]", "", x)

    if len(x) < 5:
        return None

    if x in EMPTY_TOKENS:
        return None

    return x


def norm_phone(x):
    x = clean_raw_value(x)

    if x is None:
        return None

    digits = re.sub(r"\D", "", x)

    if len(digits) == 11 and digits.startswith("8"):
        digits = "7" + digits[1:]

    if len(digits) == 10:
        digits = "7" + digits

    if len(digits) == 11 and digits.startswith("7"):
        return digits

    return None


def norm_email(x):
    x = clean_raw_value(x)

    if x is None:
        return None

    x = x.lower().strip()
    x = re.sub(r"\s+", "", x)

    if re.match(r"^[^@\s]+@[^@\s]+\.[^@\s]+$", x):
        return x

    return None


def norm_simple_category(x):
    x = clean_raw_value(x)

    if x is None:
        return None

    x = unicodedata.normalize("NFC", x)
    x = x.lower().replace("ё", "е")
    x = re.sub(r"\s+", " ", x).strip()

    return x or None

In [148]:
from pathlib import Path

print("All CSV in /content:")
all_csv = sorted(Path("/content").glob("*.csv"))
print(len(all_csv))
for f in all_csv:
    print("-", f.name)

print("\nManual CSV in /content/manual:")
manual_csv = sorted(Path("/content/manual").glob("*.csv"))
print(len(manual_csv))
for f in manual_csv:
    print("-", f.name)

print("\nraw_files:")
print(len(raw_files))
for f in raw_files:
    print("-", f.name)

All CSV in /content:
20
- Копия Анализ аудитории - ANNIVERSARY'24.csv
- Копия Анализ аудитории - Ballmer Peak'24.csv
- Копия Анализ аудитории - CSFEST'25.csv
- Копия Анализ аудитории - GLANZ.csv
- Копия Анализ аудитории - Антипосвят'23.csv
- Копия Анализ аудитории - Антипосвят'25.csv
- Копия Анализ аудитории - Бал ФКН'24.csv
- Копия Анализ аудитории - ЗВ'25.csv
- Копия Анализ аудитории - ЗВ'26.csv
- Копия Анализ аудитории - Коллаб'24.csv
- Копия Анализ аудитории - Мероприятия.csv
- Копия Анализ аудитории - Настолки'24.csv
- Копия Анализ аудитории - Нейрорейв.csv
- Копия Анализ аудитории - Посвят'23.csv
- Копия Анализ аудитории - Посвят'25.csv
- Копия Анализ аудитории - Поход'24.csv
- Копия Анализ аудитории - Поход'25.csv
- Копия Анализ аудитории - Сводная.csv
- Копия Анализ аудитории - Экватор'24.csv
- Копия Анализ аудитории - Экватор'25.csv

Manual CSV in /content/manual:
3
- attendance_corrections.csv
- event_aliases.csv
- event_metadata.csv

raw_files:
20
- Копия Анализ аудитории 

In [149]:
identity_records = []

for path in raw_files:
    inferred_event_name = clean_filename_event_name(path)
    event_info = resolve_event_name(inferred_event_name)

    df = read_csv_robust(path)
    detected = detect_identifier_columns(df)

    for row_idx, row in df.iterrows():
        fio_raw = coalesce_values(row, detected["fio"])
        telegram_raw = coalesce_values(row, detected["telegram"])
        phone_raw = coalesce_values(row, detected["phone"])
        email_raw = coalesce_values(row, detected["email"])
        group_raw = coalesce_values(row, detected["group"])
        program_raw = coalesce_values(row, detected["program"])
        course_raw = coalesce_values(row, detected["course"])

        identity_records.append({
            "source_file": path.name,
            "source_row_number": row_idx + 1,
            "inferred_event_name": inferred_event_name,
            "canonical_event_id": event_info["canonical_event_id"],
            "canonical_event_name": event_info["canonical_event_name"],
            "is_auxiliary_source": event_info["is_auxiliary_source"],

            # raw identifiers — private only
            "fio_raw": fio_raw,
            "telegram_raw": telegram_raw,
            "phone_raw": phone_raw,
            "email_raw": email_raw,
            "group_raw": group_raw,
            "program_raw": program_raw,
            "course_raw": course_raw,

            # normalized identifiers
            "fio_norm": norm_fio(fio_raw),
            "fio_signature": fio_signature(fio_raw),
            "telegram_norm": norm_telegram(telegram_raw),
            "phone_norm": norm_phone(phone_raw),
            "email_norm": norm_email(email_raw),
            "group_norm": norm_simple_category(group_raw),
            "program_norm": norm_simple_category(program_raw),
            "course_norm": norm_simple_category(course_raw),
        })

identity_records_df = pd.DataFrame(identity_records)

print("identity records:", identity_records_df.shape)

# Не выводим raw PII. Показываем только безопасную техническую сводку.
identity_records_df[
    [
        "source_file",
        "source_row_number",
        "inferred_event_name",
        "canonical_event_id",
        "is_auxiliary_source",
        "fio_norm",
        "telegram_norm",
        "phone_norm",
        "email_norm",
        "group_norm",
        "program_norm",
        "course_norm",
    ]
].head()

identity records: (11290, 21)


,source_file,source_row_number,inferred_event_name,canonical_event_id,is_auxiliary_source,fio_norm,telegram_norm,phone_norm,email_norm,group_norm,program_norm,course_norm
0,Копия Анализ аудитории - ANNIVERSARY'24.csv,1,ANNIVERSARY'24,anniversary_2024,False,бобуа андрей борисович,andruxa2oo5,None,None,None,None,None
1,Копия Анализ аудитории - ANNIVERSARY'24.csv,2,ANNIVERSARY'24,anniversary_2024,False,александрова анастасия васильевна,aji3713,None,None,None,None,None
2,Копия Анализ аудитории - ANNIVERSARY'24.csv,3,ANNIVERSARY'24,anniversary_2024,False,щербакова елизавета александровна,dreamer_1977,None,None,None,None,None
3,Копия Анализ аудитории - ANNIVERSARY'24.csv,4,ANNIVERSARY'24,anniversary_2024,False,гаврик анастасия игоревна,anaxeti,None,None,None,None,None
4,Копия Анализ аудитории - ANNIVERSARY'24.csv,5,ANNIVERSARY'24,anniversary_2024,False,щукин владислав евгеньевич,shchukin_ve,None,None,None,None,None


In [150]:
coverage_by_event = (
    identity_records_df
    .groupby(["inferred_event_name", "canonical_event_id", "is_auxiliary_source"], dropna=False)
    .agg(
        rows=("source_row_number", "count"),
        fio_present=("fio_norm", lambda s: s.notna().sum()),
        telegram_present=("telegram_norm", lambda s: s.notna().sum()),
        phone_present=("phone_norm", lambda s: s.notna().sum()),
        email_present=("email_norm", lambda s: s.notna().sum()),
        group_present=("group_norm", lambda s: s.notna().sum()),
        program_present=("program_norm", lambda s: s.notna().sum()),
        course_present=("course_norm", lambda s: s.notna().sum()),
    )
    .reset_index()
)

for col in [
    "fio_present",
    "telegram_present",
    "phone_present",
    "email_present",
    "group_present",
    "program_present",
    "course_present",
]:
    coverage_by_event[col.replace("_present", "_share")] = (
        coverage_by_event[col] / coverage_by_event["rows"]
    ).round(3)

coverage_by_event.sort_values("rows", ascending=False)


,inferred_event_name,canonical_event_id,is_auxiliary_source,rows,fio_present,telegram_present,phone_present,email_present,group_present,program_present,course_present,fio_share,telegram_share,phone_share,email_share,group_share,program_share,course_share
17,Сводная,NaN,True,5389,5074,3752,1536,0,0,3878,1698,0.942,0.696,0.285,0.000,0.000,0.720,0.315
12,Нейрорейв,neyrorave_2024,False,859,857,0,0,794,0,0,0,0.998,0.000,0.000,0.924,0.000,0.000,0.000
6,Бал ФКН'24,ball_fkn_2024,False,705,703,0,0,0,0,0,0,0.997,0.000,0.000,0.000,0.000,0.000,0.000
10,Мероприятия,NaN,True,644,0,0,0,0,0,0,0,0.000,0.000,0.000,0.000,0.000,0.000,0.000
14,Посвят'25,posvyat_2025,False,417,417,417,416,385,0,417,0,1.000,1.000,0.998,0.923,0.000,1.000,0.000
2,CSFEST'25,csfest_2025,False,371,371,369,0,371,355,0,0,1.000,0.995,0.000,1.000,0.957,0.000,0.000
13,Посвят'23,posvyat_2023,False,354,354,354,352,0,0,354,0,1.000,1.000,0.994,0.000,0.000,1.000,0.000
18,Экватор'24,ekvator_2024,False,316,270,251,0,0,0,259,259,0.854,0.794,0.000,0.000,0.000,0.820,0.820
16,Поход'25,pohod_2025,False,315,315,314,0,0,0,0,0,1.000,0.997,0.000,0.000,0.000,0.000,0.000
15,Поход'24,pohod_2024,False,287,287,287,0,0,0,0,0,1.000,1.000,0.000,0.000,0.000,0.000,0.000


In [151]:
def build_participant_key(row: pd.Series) -> tuple[str | None, str]:
    if pd.notna(row.get("telegram_norm")):
        return f"tg:{row['telegram_norm']}", "high"

    if pd.notna(row.get("phone_norm")):
        return f"phone:{row['phone_norm']}", "high"

    if pd.notna(row.get("email_norm")):
        return f"email:{row['email_norm']}", "high"

    fio = row.get("fio_signature")
    group = row.get("group_norm")
    program = row.get("program_norm")
    course = row.get("course_norm")

    if pd.notna(fio) and pd.notna(group):
        return f"fio_group:{fio}|{group}", "medium"

    if pd.notna(fio) and pd.notna(program) and pd.notna(course):
        return f"fio_program_course:{fio}|{program}|{course}", "medium"

    if pd.notna(fio) and pd.notna(program):
        return f"fio_program:{fio}|{program}", "medium"

    if pd.notna(fio):
        return f"fio_only:{fio}", "low"

    return None, "missing"


key_confidence = identity_records_df.apply(build_participant_key, axis=1)

identity_records_df["participant_key_internal"] = [x[0] for x in key_confidence]
identity_records_df["identity_confidence"] = [x[1] for x in key_confidence]

identity_records_df["has_strong_identifier"] = (
    identity_records_df["telegram_norm"].notna()
    | identity_records_df["phone_norm"].notna()
    | identity_records_df["email_norm"].notna()
)

identity_records_df["has_any_identifier"] = identity_records_df["participant_key_internal"].notna()

identity_records_df[
    [
        "inferred_event_name",
        "canonical_event_id",
        "is_auxiliary_source",
        "has_strong_identifier",
        "identity_confidence",
        "participant_key_internal",
    ]
].head()


,inferred_event_name,canonical_event_id,is_auxiliary_source,has_strong_identifier,identity_confidence,participant_key_internal
0,ANNIVERSARY'24,anniversary_2024,False,True,high,tg:andruxa2oo5
1,ANNIVERSARY'24,anniversary_2024,False,True,high,tg:aji3713
2,ANNIVERSARY'24,anniversary_2024,False,True,high,tg:dreamer_1977
3,ANNIVERSARY'24,anniversary_2024,False,True,high,tg:anaxeti
4,ANNIVERSARY'24,anniversary_2024,False,True,high,tg:shchukin_ve


In [152]:
# Для реального проекта лучше задать соль через переменную окружения:
# os.environ["IDENTITY_SALT"] = "your_secret_salt"
#
# В публичный GitHub соль класть нельзя.

IDENTITY_SALT = os.getenv("IDENTITY_SALT", "local_dev_salt_do_not_publish")


def hmac_sha256(value: str | None, salt: str = IDENTITY_SALT) -> str | None:
    if value is None or pd.isna(value):
        return None

    return hmac.new(
        salt.encode("utf-8"),
        str(value).encode("utf-8"),
        hashlib.sha256
    ).hexdigest()


identity_records_df["participant_hash_private"] = identity_records_df["participant_key_internal"].apply(
    hmac_sha256
)

identity_records_df[["participant_key_internal", "participant_hash_private", "identity_confidence"]].head()

,participant_key_internal,participant_hash_private,identity_confidence
0,tg:andruxa2oo5,90a40ead6c9dd5706bd26764f314f2f890aa251b56cd61aa0d8cac4e2fa2772f,high
1,tg:aji3713,b087574bb17161d91aed946a0f0947920db42da72f0f9193fb48a83d50db8336,high
2,tg:dreamer_1977,614cfb3ae9effd4c6395dc2af3d52753e9dc2effec710b567333b4cd807dca73,high
3,tg:anaxeti,ab726f8a2d53b286749447f4a97a08bb3057c2b334bf6a48d7c8f0097626ad41,high
4,tg:shchukin_ve,e94aa290a374312cbad1f14d39ccf19c5e4dd4851bb9466b15c3934cbfabca22,high


In [153]:
identity_summary = {
    "total_rows": len(identity_records_df),
    "rows_with_any_identifier": int(identity_records_df["has_any_identifier"].sum()),
    "rows_with_strong_identifier": int(identity_records_df["has_strong_identifier"].sum()),
    "rows_missing_identifier": int((~identity_records_df["has_any_identifier"]).sum()),
    "unique_participant_keys": int(identity_records_df["participant_key_internal"].nunique(dropna=True)),
    "unique_private_hashes": int(identity_records_df["participant_hash_private"].nunique(dropna=True)),
}

identity_summary_df = pd.DataFrame([identity_summary])

identity_summary_df


,total_rows,rows_with_any_identifier,rows_with_strong_identifier,rows_missing_identifier,unique_participant_keys,unique_private_hashes
0,11290,10193,7764,1097,5573,5573


In [154]:
confidence_distribution = (
    identity_records_df
    .groupby("identity_confidence", dropna=False)
    .agg(
        rows=("source_row_number", "count"),
        unique_participant_keys=("participant_key_internal", "nunique"),
    )
    .reset_index()
    .sort_values("rows", ascending=False)
)

confidence_distribution["row_share"] = (
    confidence_distribution["rows"] / confidence_distribution["rows"].sum()
).round(3)

confidence_distribution


,identity_confidence,rows,unique_participant_keys,row_share
0,high,7764,3231,0.688
2,medium,1560,1527,0.138
3,missing,1097,0,0.097
1,low,869,815,0.077


In [155]:
identity_quality_by_event = (
    identity_records_df
    .groupby(["inferred_event_name", "canonical_event_id", "is_auxiliary_source"], dropna=False)
    .agg(
        rows=("source_row_number", "count"),
        rows_with_any_identifier=("has_any_identifier", "sum"),
        rows_with_strong_identifier=("has_strong_identifier", "sum"),
        unique_participant_keys=("participant_key_internal", "nunique"),
    )
    .reset_index()
)

identity_quality_by_event["any_identifier_share"] = (
    identity_quality_by_event["rows_with_any_identifier"] / identity_quality_by_event["rows"]
).round(3)

identity_quality_by_event["strong_identifier_share"] = (
    identity_quality_by_event["rows_with_strong_identifier"] / identity_quality_by_event["rows"]
).round(3)

identity_quality_by_event.sort_values("strong_identifier_share")


,inferred_event_name,canonical_event_id,is_auxiliary_source,rows,rows_with_any_identifier,rows_with_strong_identifier,unique_participant_keys,any_identifier_share,strong_identifier_share
6,Бал ФКН'24,ball_fkn_2024,False,705,703,0,699,0.997,0.000
9,Коллаб'24,collab_2024,False,276,0,0,0,0.000,0.000
10,Мероприятия,NaN,True,644,0,0,0,0.000,0.000
3,GLANZ,glanz_2024,False,281,168,164,166,0.598,0.584
17,Сводная,NaN,True,5389,5380,3752,3918,0.998,0.696
18,Экватор'24,ekvator_2024,False,316,270,251,264,0.854,0.794
12,Нейрорейв,neyrorave_2024,False,859,857,794,809,0.998,0.924
8,ЗВ'26,zv_2026,False,69,69,64,68,1.000,0.928
0,ANNIVERSARY'24,anniversary_2024,False,86,86,83,81,1.000,0.965
1,Ballmer Peak'24,ballmer_peak_2024,False,150,146,146,143,0.973,0.973


In [156]:
# Проверяем конфликты только на event-level источниках.
# Auxiliary sources вроде "Сводная" и "Мероприятия" не используем для оценки конфликтов,
# потому что они дублируют raw-данные и могут искусственно завышать число конфликтов.
#
# Конфликт = один и тот же сильный идентификатор Telegram / phone / email
# связан с несколькими разными fio_signature.

event_identity_df = identity_records_df[
    identity_records_df["is_auxiliary_source"] == False
].copy()

strong_id_long = []

for id_col, id_type in [
    ("telegram_norm", "telegram"),
    ("phone_norm", "phone"),
    ("email_norm", "email"),
]:
    tmp = event_identity_df[
        event_identity_df[id_col].notna()
    ][[id_col, "fio_signature", "inferred_event_name", "source_file"]].copy()

    tmp = tmp.rename(columns={id_col: "identifier_value"})
    tmp["identifier_type"] = id_type

    strong_id_long.append(tmp)

strong_id_long_df = (
    pd.concat(strong_id_long, ignore_index=True)
    if strong_id_long
    else pd.DataFrame()
)

if not strong_id_long_df.empty:
    strong_id_conflicts = (
        strong_id_long_df
        .groupby(["identifier_type", "identifier_value"], dropna=False)
        .agg(
            n_fio=("fio_signature", "nunique"),
            n_rows=("fio_signature", "size"),
            events=(
                "inferred_event_name",
                lambda s: ", ".join(sorted(set(s.dropna().astype(str)))[:10])
            ),
        )
        .reset_index()
    )

    strong_id_conflicts = strong_id_conflicts[
        strong_id_conflicts["n_fio"] > 1
    ].sort_values(["n_fio", "n_rows"], ascending=False)
else:
    strong_id_conflicts = pd.DataFrame(
        columns=[
            "identifier_type",
            "identifier_value",
            "n_fio",
            "n_rows",
            "events",
        ]
    )

print("potential strong id conflicts, event-level only:", len(strong_id_conflicts))

# Безопасный preview без самих Telegram / phone / email.
strong_id_conflicts_safe = strong_id_conflicts.drop(
    columns=["identifier_value"],
    errors="ignore"
)

display(strong_id_conflicts_safe.head(20))

potential strong id conflicts, event-level only: 353


,identifier_type,n_fio,n_rows,events
2801,telegram,4,6,"GLANZ, Антипосвят'25, Посвят'23, Поход'24, Поход'25, Экватор'25"
3079,telegram,4,6,"Ballmer Peak'24, GLANZ, Антипосвят'25, ЗВ'26, Посвят'23, Экватор'25"
3668,telegram,4,6,"GLANZ, Антипосвят'23, Настолки'24, Поход'24, Экватор'24, Экватор'25"
2579,telegram,4,4,ANNIVERSARY'24
3899,telegram,4,4,"GLANZ, Антипосвят'23, Экватор'24, Экватор'25"
3657,telegram,3,8,"Ballmer Peak'24, GLANZ, Антипосвят'23, Антипосвят'25, ЗВ'25, Поход'24, Поход'25, Экватор'24"
3697,telegram,3,8,"ANNIVERSARY'24, CSFEST'25, GLANZ, Посвят'23, Поход'24, Поход'25, Экватор'24, Экватор'25"
2725,telegram,3,7,"CSFEST'25, GLANZ, Антипосвят'25, ЗВ'26, Посвят'23, Экватор'24, Экватор'25"
4107,telegram,3,7,"GLANZ, Антипосвят'23, Антипосвят'25, ЗВ'25, ЗВ'26, Поход'25, Экватор'25"
2779,telegram,3,6,"GLANZ, Антипосвят'23, Поход'25, Экватор'24, Экватор'25"


In [157]:
# Потенциальные дубли по ФИО проверяем только на event-level источниках.
# Используем fio_signature, чтобы "Иванов Иван" и "Иван Иванов" считались одним вариантом ФИО.

fio_potential_duplicates = (
    event_identity_df[
        event_identity_df["fio_signature"].notna()
        & event_identity_df["has_strong_identifier"]
    ]
    .groupby("fio_signature", dropna=False)
    .agg(
        n_telegram=("telegram_norm", "nunique"),
        n_phone=("phone_norm", "nunique"),
        n_email=("email_norm", "nunique"),
        n_rows=("source_row_number", "count"),
        events=("inferred_event_name", lambda s: ", ".join(sorted(set(s.dropna().astype(str)))[:10])),
    )
    .reset_index()
)

fio_potential_duplicates["n_strong_ids"] = (
    fio_potential_duplicates["n_telegram"]
    + fio_potential_duplicates["n_phone"]
    + fio_potential_duplicates["n_email"]
)

fio_potential_duplicates = fio_potential_duplicates[
    fio_potential_duplicates["n_strong_ids"] > 1
].sort_values(["n_strong_ids", "n_rows"], ascending=False)

print(
    "potential duplicate names with multiple strong ids, event-level only:",
    len(fio_potential_duplicates)
)

# Это private output: в публичный README / GitHub-вывод с fio_signature не вставляем.
display(fio_potential_duplicates.head(20))

potential duplicate names with multiple strong ids, event-level only: 1424


,fio_signature,n_telegram,n_phone,n_email,n_rows,events,n_strong_ids
2489,майя рабинович,1,1,3,6,"CSFEST'25, Антипосвят'25, Нейрорейв, Поход'25, Экватор'25",5
242,александрович грек федор,2,1,2,5,"CSFEST'25, GLANZ, Антипосвят'25, ЗВ'25, Нейрорейв",5
1780,горячев иван сергеевич,2,1,2,5,"CSFEST'25, GLANZ, Антипосвят'25, Нейрорейв, Посвят'23",5
2396,константинович пахуров федор,2,1,2,4,"CSFEST'25, GLANZ, Антипосвят'25, Нейрорейв",5
935,анисимова варвара романовна,1,1,2,9,"CSFEST'25, Настолки'24, Нейрорейв, Посвят'23, Поход'25",4
1879,даниэль серегина фабиан фрейре,2,1,1,9,"GLANZ, Антипосвят'25, Настолки'24, Нейрорейв, Посвят'23, Поход'24, Поход'25, Экватор'25",4
512,алексеевна иванова карина,2,1,1,8,"CSFEST'25, Антипосвят'23, Нейрорейв, Посвят'23, Экватор'24",4
1051,арина дмитриевна сиротинкина,2,1,1,7,"ANNIVERSARY'24, CSFEST'25, GLANZ, Антипосвят'25, Настолки'24, Посвят'23, Поход'24",4
1673,гагарина ульяна юрьевна,2,1,1,6,"CSFEST'25, Антипосвят'23, Нейрорейв, Посвят'23, Поход'25",4
2582,муратович тимур шокаров,1,1,2,6,"CSFEST'25, Антипосвят'23, Настолки'24, Нейрорейв, Поход'24",4


In [158]:
# Помечаем participant_key_internal, построенные на конфликтных strong identifiers.

if not strong_id_conflicts.empty:
    conflict_identifier_values = set(strong_id_conflicts["identifier_value"].dropna().astype(str))
else:
    conflict_identifier_values = set()


def has_identity_conflict(row: pd.Series) -> bool:
    for col in ["telegram_norm", "phone_norm", "email_norm"]:
        value = row.get(col)
        if pd.notna(value) and str(value) in conflict_identifier_values:
            return True
    return False


identity_records_df["identity_conflict_flag"] = identity_records_df.apply(
    has_identity_conflict,
    axis=1
)

identity_conflict_summary = (
    identity_records_df
    .groupby(["is_auxiliary_source", "identity_conflict_flag"], dropna=False)
    .agg(
        rows=("source_row_number", "count"),
        unique_participant_keys=("participant_key_internal", "nunique"),
    )
    .reset_index()
)

display(identity_conflict_summary)

,is_auxiliary_source,identity_conflict_flag,rows,unique_participant_keys
0,False,False,4170,3303
1,False,True,1087,336
2,True,False,4863,3611
3,True,True,1170,310


In [159]:
identity_records_safe_preview = identity_records_df[
    [
        "source_file",
        "source_row_number",
        "inferred_event_name",
        "canonical_event_id",
        "is_auxiliary_source",
        "has_strong_identifier",
        "has_any_identifier",
        "identity_confidence",
        "identity_conflict_flag",
        "participant_hash_private",
    ]
].copy()

identity_records_safe_preview.head()

,source_file,source_row_number,inferred_event_name,canonical_event_id,is_auxiliary_source,has_strong_identifier,has_any_identifier,identity_confidence,identity_conflict_flag,participant_hash_private
0,Копия Анализ аудитории - ANNIVERSARY'24.csv,1,ANNIVERSARY'24,anniversary_2024,False,True,True,high,False,90a40ead6c9dd5706bd26764f314f2f890aa251b56cd61aa0d8cac4e2fa2772f
1,Копия Анализ аудитории - ANNIVERSARY'24.csv,2,ANNIVERSARY'24,anniversary_2024,False,True,True,high,False,b087574bb17161d91aed946a0f0947920db42da72f0f9193fb48a83d50db8336
2,Копия Анализ аудитории - ANNIVERSARY'24.csv,3,ANNIVERSARY'24,anniversary_2024,False,True,True,high,False,614cfb3ae9effd4c6395dc2af3d52753e9dc2effec710b567333b4cd807dca73
3,Копия Анализ аудитории - ANNIVERSARY'24.csv,4,ANNIVERSARY'24,anniversary_2024,False,True,True,high,True,ab726f8a2d53b286749447f4a97a08bb3057c2b334bf6a48d7c8f0097626ad41
4,Копия Анализ аудитории - ANNIVERSARY'24.csv,5,ANNIVERSARY'24,anniversary_2024,False,True,True,high,False,e94aa290a374312cbad1f14d39ccf19c5e4dd4851bb9466b15c3934cbfabca22


In [160]:
identity_records_private_path = IDENTITY_DIR / "identity_records_private.csv"
identity_summary_path = IDENTITY_DIR / "identity_summary.csv"
identity_quality_by_event_path = IDENTITY_DIR / "identity_quality_by_event.csv"
confidence_distribution_path = IDENTITY_DIR / "confidence_distribution.csv"
strong_id_conflicts_path = IDENTITY_DIR / "strong_id_conflicts_private.csv"
fio_duplicates_path = IDENTITY_DIR / "fio_potential_duplicates_private.csv"

identity_records_df.to_csv(identity_records_private_path, index=False)
identity_summary_df.to_csv(identity_summary_path, index=False)
identity_quality_by_event.to_csv(identity_quality_by_event_path, index=False)
confidence_distribution.to_csv(confidence_distribution_path, index=False)
strong_id_conflicts.to_csv(strong_id_conflicts_path, index=False)
fio_potential_duplicates.to_csv(fio_duplicates_path, index=False)

print("saved:", identity_records_private_path)
print("saved:", identity_summary_path)
print("saved:", identity_quality_by_event_path)
print("saved:", confidence_distribution_path)
print("saved:", strong_id_conflicts_path)
print("saved:", fio_duplicates_path)

saved: /content/data/interim_private/identity/identity_records_private.csv
saved: /content/data/interim_private/identity/identity_summary.csv
saved: /content/data/interim_private/identity/identity_quality_by_event.csv
saved: /content/data/interim_private/identity/confidence_distribution.csv
saved: /content/data/interim_private/identity/strong_id_conflicts_private.csv
saved: /content/data/interim_private/identity/fio_potential_duplicates_private.csv


In [161]:
print("total rows:", len(identity_records_df))
print("unique participant keys:", identity_records_df["participant_key_internal"].nunique(dropna=True))
print("rows with any identifier:", int(identity_records_df["has_any_identifier"].sum()))
print("rows with strong identifier:", int(identity_records_df["has_strong_identifier"].sum()))
print("rows missing identifier:", int((~identity_records_df["has_any_identifier"]).sum()))

print()
print("event-level rows:", len(event_identity_df))
print("event-level unique participant keys:", event_identity_df["participant_key_internal"].nunique(dropna=True))

print()
print("confidence distribution:")
display(confidence_distribution)

print()
print("identity quality by event:")
display(identity_quality_by_event.sort_values("strong_identifier_share"))

print()
print("potential strong id conflicts, event-level only:", len(strong_id_conflicts))
print("potential duplicate names, event-level only:", len(fio_potential_duplicates))

print()
print("identity conflict flag summary:")
display(identity_conflict_summary)

total rows: 11290
unique participant keys: 5573
rows with any identifier: 10193
rows with strong identifier: 7764
rows missing identifier: 1097

event-level rows: 5257
event-level unique participant keys: 3636

confidence distribution:


,identity_confidence,rows,unique_participant_keys,row_share
0,high,7764,3231,0.688
2,medium,1560,1527,0.138
3,missing,1097,0,0.097
1,low,869,815,0.077



identity quality by event:


,inferred_event_name,canonical_event_id,is_auxiliary_source,rows,rows_with_any_identifier,rows_with_strong_identifier,unique_participant_keys,any_identifier_share,strong_identifier_share
6,Бал ФКН'24,ball_fkn_2024,False,705,703,0,699,0.997,0.000
9,Коллаб'24,collab_2024,False,276,0,0,0,0.000,0.000
10,Мероприятия,NaN,True,644,0,0,0,0.000,0.000
3,GLANZ,glanz_2024,False,281,168,164,166,0.598,0.584
17,Сводная,NaN,True,5389,5380,3752,3918,0.998,0.696
18,Экватор'24,ekvator_2024,False,316,270,251,264,0.854,0.794
12,Нейрорейв,neyrorave_2024,False,859,857,794,809,0.998,0.924
8,ЗВ'26,zv_2026,False,69,69,64,68,1.000,0.928
0,ANNIVERSARY'24,anniversary_2024,False,86,86,83,81,1.000,0.965
1,Ballmer Peak'24,ballmer_peak_2024,False,150,146,146,143,0.973,0.973



potential strong id conflicts, event-level only: 353
potential duplicate names, event-level only: 1424

identity conflict flag summary:


,is_auxiliary_source,identity_conflict_flag,rows,unique_participant_keys
0,False,False,4170,3303
1,False,True,1087,336
2,True,False,4863,3611
3,True,True,1170,310


## Итоги identity resolution

После нормализации идентификаторов получен private identity layer.

### Ключевые результаты

- Всего обработано 11 290 строк.
- Построено 6 387 уникальных `participant_key_internal` до финальной фильтрации.
- 68.8% строк имеют high-confidence identity на основе Telegram / телефона / email.
- 13.8% строк имеют medium-confidence identity.
- 7.7% строк имеют low-confidence identity.
- 9.7% строк остались без пригодного идентификатора после более строгой фильтрации колонок.

### Data quality decisions

- `Сводная.csv` и `Мероприятия.csv` рассматриваются как auxiliary sources и не используются для оценки конфликтов identity.
- Для `Коллаб'24` нужна отдельная special-loader логика, потому что данные представлены через `Unnamed`-структуру.
- Для `Бал ФКН'24` почти все строки имеют ФИО, но нет сильных идентификаторов, поэтому событие нужно осторожно использовать в cross-event retention.
- Найдено 353 потенциальных конфликта сильных идентификаторов на event-level данных.
- Найдено 1424 потенциальных случая, где одно ФИО связано с несколькими strong identifiers.

### Методологическое решение

Конфликтные strong identifiers не исправляются вручную без внешнего подтверждения.  
Вместо этого они помечаются через `identity_conflict_flag`.

Для будущих retention и journey-метрик будет использоваться clean identity layer:

```python
clean_identity_df = identity_records_df[
    (identity_records_df["is_auxiliary_source"] == False)
    & (identity_records_df["identity_conflict_flag"] == False)
    & (identity_records_df["identity_confidence"].isin(["high", "medium"]))
].copy()

In [162]:
print("total rows:", len(identity_records_df))
print("unique participant keys:", identity_records_df["participant_key_internal"].nunique(dropna=True))
print("rows with any identifier:", int(identity_records_df["has_any_identifier"].sum()))
print("rows with strong identifier:", int(identity_records_df["has_strong_identifier"].sum()))
print("rows missing identifier:", int((~identity_records_df["has_any_identifier"]).sum()))

print()
print("confidence distribution:")
display(confidence_distribution)

print()
print("identity quality by event:")
display(identity_quality_by_event.sort_values("strong_identifier_share"))

print()
print("potential strong id conflicts:", len(strong_id_conflicts))
print("potential duplicate names:", len(fio_potential_duplicates))

total rows: 11290
unique participant keys: 5573
rows with any identifier: 10193
rows with strong identifier: 7764
rows missing identifier: 1097

confidence distribution:


,identity_confidence,rows,unique_participant_keys,row_share
0,high,7764,3231,0.688
2,medium,1560,1527,0.138
3,missing,1097,0,0.097
1,low,869,815,0.077



identity quality by event:


,inferred_event_name,canonical_event_id,is_auxiliary_source,rows,rows_with_any_identifier,rows_with_strong_identifier,unique_participant_keys,any_identifier_share,strong_identifier_share
6,Бал ФКН'24,ball_fkn_2024,False,705,703,0,699,0.997,0.000
9,Коллаб'24,collab_2024,False,276,0,0,0,0.000,0.000
10,Мероприятия,NaN,True,644,0,0,0,0.000,0.000
3,GLANZ,glanz_2024,False,281,168,164,166,0.598,0.584
17,Сводная,NaN,True,5389,5380,3752,3918,0.998,0.696
18,Экватор'24,ekvator_2024,False,316,270,251,264,0.854,0.794
12,Нейрорейв,neyrorave_2024,False,859,857,794,809,0.998,0.924
8,ЗВ'26,zv_2026,False,69,69,64,68,1.000,0.928
0,ANNIVERSARY'24,anniversary_2024,False,86,86,83,81,1.000,0.965
1,Ballmer Peak'24,ballmer_peak_2024,False,150,146,146,143,0.973,0.973



potential strong id conflicts: 353
potential duplicate names: 1424


In [163]:
display(confidence_distribution)

display(identity_quality_by_event.sort_values("strong_identifier_share"))

print("potential strong id conflicts:", len(strong_id_conflicts))
print("potential duplicate names:", len(fio_potential_duplicates))

,identity_confidence,rows,unique_participant_keys,row_share
0,high,7764,3231,0.688
2,medium,1560,1527,0.138
3,missing,1097,0,0.097
1,low,869,815,0.077


,inferred_event_name,canonical_event_id,is_auxiliary_source,rows,rows_with_any_identifier,rows_with_strong_identifier,unique_participant_keys,any_identifier_share,strong_identifier_share
6,Бал ФКН'24,ball_fkn_2024,False,705,703,0,699,0.997,0.000
9,Коллаб'24,collab_2024,False,276,0,0,0,0.000,0.000
10,Мероприятия,NaN,True,644,0,0,0,0.000,0.000
3,GLANZ,glanz_2024,False,281,168,164,166,0.598,0.584
17,Сводная,NaN,True,5389,5380,3752,3918,0.998,0.696
18,Экватор'24,ekvator_2024,False,316,270,251,264,0.854,0.794
12,Нейрорейв,neyrorave_2024,False,859,857,794,809,0.998,0.924
8,ЗВ'26,zv_2026,False,69,69,64,68,1.000,0.928
0,ANNIVERSARY'24,anniversary_2024,False,86,86,83,81,1.000,0.965
1,Ballmer Peak'24,ballmer_peak_2024,False,150,146,146,143,0.973,0.973


potential strong id conflicts: 353
potential duplicate names: 1424


In [164]:
display(identifier_column_inventory_df)
conflict_summary_by_type = (
    strong_id_conflicts
    .groupby("identifier_type")
    .agg(
        conflicts=("identifier_type", "count"),
        total_rows=("n_rows", "sum"),
        max_fio_per_identifier=("n_fio", "max"),
        avg_fio_per_identifier=("n_fio", "mean"),
    )
    .reset_index()
)

display(conflict_summary_by_type)

conflict_events = (
    strong_id_conflicts
    .assign(events_list=strong_id_conflicts["events"].str.split(", "))
    .explode("events_list")
    .groupby(["identifier_type", "events_list"])
    .agg(conflicts=("identifier_type", "count"))
    .reset_index()
    .sort_values("conflicts", ascending=False)
)

display(conflict_events.head(30))

,source_file,inferred_event_name,fio_columns,fio_n_columns,telegram_columns,telegram_n_columns,phone_columns,phone_n_columns,email_columns,email_n_columns,group_columns,group_n_columns,program_columns,program_n_columns,course_columns,course_n_columns
0,Копия Анализ аудитории - ANNIVERSARY'24.csv,ANNIVERSARY'24,ФИО,1,Телеграм,1,,0,,0,,0,,0,,0
1,Копия Анализ аудитории - Ballmer Peak'24.csv,Ballmer Peak'24,,0,Tg @,1,,0,,0,,0,Направление,1,,0
2,Копия Анализ аудитории - CSFEST'25.csv,CSFEST'25,Фамилия | Имя | Отчество,3,Телеграм,1,,0,Email,1,Группа,1,,0,,0
3,Копия Анализ аудитории - GLANZ.csv,GLANZ,ФИО,1,Телеграм,1,,0,,0,,0,Факультет,1,,0
4,Копия Анализ аудитории - Антипосвят'23.csv,Антипосвят'23,ФИО,1,Телеграм,1,Телефон,1,,0,,0,Направление,1,Курс,1
5,Копия Анализ аудитории - Антипосвят'25.csv,Антипосвят'25,ФИО,1,Телеграм,1,Телефон,1,,0,,0,,0,Курс,1
6,Копия Анализ аудитории - Бал ФКН'24.csv,Бал ФКН'24,ФИО,1,,0,,0,,0,,0,,0,,0
7,Копия Анализ аудитории - ЗВ'25.csv,ЗВ'25,ФИО,1,Телеграм,1,Телефон,1,,0,,0,Программа,1,Курс,1
8,Копия Анализ аудитории - ЗВ'26.csv,ЗВ'26,Имя | Фамилия,2,Телеграм,1,,0,,0,,0,,0,,0
9,Копия Анализ аудитории - Коллаб'24.csv,Коллаб'24,,0,,0,,0,,0,,0,,0,,0


,identifier_type,conflicts,total_rows,max_fio_per_identifier,avg_fio_per_identifier
0,email,19,43,2,2.00000
1,phone,31,68,2,2.00000
2,telegram,303,1048,4,2.09901


,identifier_type,events_list,conflicts
21,telegram,Экватор'25,161
20,telegram,Экватор'24,134
12,telegram,Антипосвят'25,105
9,telegram,CSFEST'25,92
10,telegram,GLANZ,91
19,telegram,Поход'25,71
11,telegram,Антипосвят'23,68
16,telegram,Посвят'23,66
18,telegram,Поход'24,49
15,telegram,Настолки'24,43


In [165]:
display(identifier_column_inventory_df)

display(confidence_distribution)

display(identity_quality_by_event.sort_values("strong_identifier_share"))

print("potential strong id conflicts:", len(strong_id_conflicts))
print("potential duplicate names:", len(fio_potential_duplicates))

,source_file,inferred_event_name,fio_columns,fio_n_columns,telegram_columns,telegram_n_columns,phone_columns,phone_n_columns,email_columns,email_n_columns,group_columns,group_n_columns,program_columns,program_n_columns,course_columns,course_n_columns
0,Копия Анализ аудитории - ANNIVERSARY'24.csv,ANNIVERSARY'24,ФИО,1,Телеграм,1,,0,,0,,0,,0,,0
1,Копия Анализ аудитории - Ballmer Peak'24.csv,Ballmer Peak'24,,0,Tg @,1,,0,,0,,0,Направление,1,,0
2,Копия Анализ аудитории - CSFEST'25.csv,CSFEST'25,Фамилия | Имя | Отчество,3,Телеграм,1,,0,Email,1,Группа,1,,0,,0
3,Копия Анализ аудитории - GLANZ.csv,GLANZ,ФИО,1,Телеграм,1,,0,,0,,0,Факультет,1,,0
4,Копия Анализ аудитории - Антипосвят'23.csv,Антипосвят'23,ФИО,1,Телеграм,1,Телефон,1,,0,,0,Направление,1,Курс,1
5,Копия Анализ аудитории - Антипосвят'25.csv,Антипосвят'25,ФИО,1,Телеграм,1,Телефон,1,,0,,0,,0,Курс,1
6,Копия Анализ аудитории - Бал ФКН'24.csv,Бал ФКН'24,ФИО,1,,0,,0,,0,,0,,0,,0
7,Копия Анализ аудитории - ЗВ'25.csv,ЗВ'25,ФИО,1,Телеграм,1,Телефон,1,,0,,0,Программа,1,Курс,1
8,Копия Анализ аудитории - ЗВ'26.csv,ЗВ'26,Имя | Фамилия,2,Телеграм,1,,0,,0,,0,,0,,0
9,Копия Анализ аудитории - Коллаб'24.csv,Коллаб'24,,0,,0,,0,,0,,0,,0,,0


,identity_confidence,rows,unique_participant_keys,row_share
0,high,7764,3231,0.688
2,medium,1560,1527,0.138
3,missing,1097,0,0.097
1,low,869,815,0.077


,inferred_event_name,canonical_event_id,is_auxiliary_source,rows,rows_with_any_identifier,rows_with_strong_identifier,unique_participant_keys,any_identifier_share,strong_identifier_share
6,Бал ФКН'24,ball_fkn_2024,False,705,703,0,699,0.997,0.000
9,Коллаб'24,collab_2024,False,276,0,0,0,0.000,0.000
10,Мероприятия,NaN,True,644,0,0,0,0.000,0.000
3,GLANZ,glanz_2024,False,281,168,164,166,0.598,0.584
17,Сводная,NaN,True,5389,5380,3752,3918,0.998,0.696
18,Экватор'24,ekvator_2024,False,316,270,251,264,0.854,0.794
12,Нейрорейв,neyrorave_2024,False,859,857,794,809,0.998,0.924
8,ЗВ'26,zv_2026,False,69,69,64,68,1.000,0.928
0,ANNIVERSARY'24,anniversary_2024,False,86,86,83,81,1.000,0.965
1,Ballmer Peak'24,ballmer_peak_2024,False,150,146,146,143,0.973,0.973


potential strong id conflicts: 353
potential duplicate names: 1424


In [166]:
clean_identity_df = identity_records_df[
    (identity_records_df["is_auxiliary_source"] == False)
    & (identity_records_df["identity_conflict_flag"] == False)
    & (identity_records_df["identity_confidence"].isin(["high", "medium"]))
].copy()

clean_identity_summary = pd.DataFrame([{
    "event_level_rows": len(event_identity_df),
    "clean_identity_rows": len(clean_identity_df),
    "clean_identity_row_share": round(len(clean_identity_df) / len(event_identity_df), 3),
    "event_level_unique_keys": event_identity_df["participant_key_internal"].nunique(dropna=True),
    "clean_unique_keys": clean_identity_df["participant_key_internal"].nunique(dropna=True),
}])

display(clean_identity_summary)

,event_level_rows,clean_identity_rows,clean_identity_row_share,event_level_unique_keys,clean_unique_keys
0,5257,2933,0.558,3636,2527
